In [ ]:
import re 
import random
from rdkit import Chem
from typing import Optional
from collections import Counter

In [ ]:
def shift_chuckles(chuckles: str) -> Optional[str]:
    def _extract_unpaired_num(sequence_part: str) -> str:
        # Find all ring numbers (single digits or %XX format)
        ring_numbers = re.findall(r'%\d+|\d', sequence_part)
        
        # Count occurrences of each ring number
        counts = Counter(ring_numbers)
        
        # Find the ring number that appears an odd number of times
        for ring_num, count in counts.items():
            if count % 2 != 0:  # Odd count indicates unpaired ring
                return ring_num
        
        return ''
    
    # Split the sequence into individual units
    chuckles_list = chuckles.split('|')
    shift = random.randint(0, len(chuckles_list)-1)
    
    # Extract unpaired ring numbers from first and last units
    # These represent the cyclization points of the peptide
    first_number = _extract_unpaired_num(chuckles_list[0])
    last_number = _extract_unpaired_num(chuckles_list[-1])
    
    # If ring numbers don't match, this isn't a proper cyclic peptide
    if first_number != last_number:
        first_number, last_number = '', ''
    
    # Remove ring numbers from terminal units before shifting
    chuckles_list[0] = chuckles_list[0].replace(first_number, '')
    chuckles_list[-1] = chuckles_list[-1].replace(first_number, '')
    
    # Perform the cyclic shift (wraps around list length)
    shift = shift % len(chuckles_list)
    chuckles_shifted = chuckles_list[-shift:] + chuckles_list[:-shift]
    
    # Restore ring numbers to new terminal positions
    if first_number:
        # Add ring number after first character of new first unit
        chuckles_shifted[0] = (chuckles_shifted[0][:1] + 
                              first_number + 
                              chuckles_shifted[0][1:])
        
        # Add ring number to carbonyl group in new last unit
        chuckles_shifted[-1] = re.sub(r'C\(=O\)$', 
                                     f'C{last_number}(=O)', 
                                     chuckles_shifted[-1])
    
    # Validate the shifted sequence using RDKit
    combined_smiles = ''.join(chuckles_shifted)
    if Chem.MolFromSmiles(combined_smiles):
        return '|'.join(chuckles_shifted)
    else:
        return chuckles

In [ ]:
shift_chuckles('N2[C@@H](CC(C)C)C(=O)|N1[C@@H](CCC1)C(=O)|N[C@@H](Cc1c(Cl)ccnc1C(=O)O)C(=O)|N[C@@H](CCCCN)C(=O)|N[C@@H](CCCNC(=N)N)C(=O)|N[C@@H](CC(C)C)C2(=O)')

In [ ]:
from rdkit import Chem
from rdkit.Chem import BRICS

smiles = "N[C@@H](Cc1c(Cl)ccnc1C(=O)O)C(=O)"  # Aspirin
mol = Chem.MolFromSmiles(smiles)

# Generate BRICS fragments
frags = BRICS.BRICSDecompose(mol)
print(frags)


In [ ]:
class SMILESTokenizer:
    """Deals with the tokenization and untokenization of SMILES."""

    REGEXPS = {
        "brackets": re.compile(r"(\[[^\]]*\])"),
        "2_ring_nums": re.compile(r"(%\d{2})"),
        "brcl": re.compile(r"(Br|Cl)")
    }
    REGEXP_ORDER = ["brackets", "2_ring_nums", "brcl"]

    def tokenize(self, data, with_begin_and_end=True):
        """Tokenizes a SMILES string."""
        def split_by(data, regexps):
            if not regexps:
                return list(data)
            regexp = self.REGEXPS[regexps[0]]
            splitted = regexp.split(data)
            tokens = []
            for i, split in enumerate(splitted):
                if i % 2 == 0:
                    tokens += split_by(split, regexps[1:])
                else:
                    tokens.append(split)
            return tokens

        tokens = split_by(data, self.REGEXP_ORDER)
        if with_begin_and_end:
            tokens = ["^"] + tokens + ["$"]
        return tokens

    def untokenize(self, tokens):
        """Untokenizes a SMILES string."""
        smi = ""
        for i, token in enumerate(tokens):
            if token == "$":
                break
            if token != "^" or (token == "^" and i != 0):
                smi += token
        return smi

In [ ]:
t = SMILESTokenizer()

In [ ]:
t.tokenize('[1]CCCCCC[/1]')